# SME-DataTrust BI — configuración
Ejecuta los notebooks del 01 al 06 en orden y dentro de la misma sesión de Colab. Si tu carpeta del proyecto está en otra ruta, modifica `PROJECT_DIR`.

In [ ]:
from pathlib import Path
import sys
candidates=[Path("/content/BI-PYMES-ML"),Path("/content/SME-DataTrust-BI"),Path.cwd()]
PROJECT_DIR=next((p for p in candidates if (p/"src").exists()),Path.cwd())
if not (PROJECT_DIR/"src").exists():
    raise FileNotFoundError("No se encontró src/. Ajusta PROJECT_DIR a la raíz del repositorio.")
sys.path.insert(0,str(PROJECT_DIR))
for d in ["data/raw","data/processed","data/experimental","results/figures","results/metrics"]:
    (PROJECT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Proyecto:",PROJECT_DIR)

## 1. Descarga y exploración
Dataset: Online Retail II, UCI Machine Learning Repository. El archivo se descarga localmente y no se versiona en Git.

In [ ]:
import pandas as pd
from src.preprocessing import download_online_retail_ii,load_retail_excel,quality_profile
xlsx=download_online_retail_ii(PROJECT_DIR/"data/raw")
df=load_retail_excel(xlsx)
print("Archivo:",xlsx,"\nDimensiones:",df.shape)
display(df.head())
display(df.dtypes)

In [ ]:
profile,checks=quality_profile(df)
display(profile)
print(checks)
profile.to_csv(PROJECT_DIR/"results/metrics/initial_data_profile.csv")
pd.Series(checks).to_json(PROJECT_DIR/"results/metrics/initial_quality_checks.json",indent=2)
df.to_pickle(PROJECT_DIR/"data/processed/retail_lines.pkl")

In [ ]:
import matplotlib.pyplot as plt
missing=(df.isna().mean()*100).sort_values(ascending=False)
ax=missing.plot(kind="bar",figsize=(10,4),title="Porcentaje de valores faltantes")
ax.set_ylabel("%"); plt.tight_layout()
plt.savefig(PROJECT_DIR/"results/figures/missing_values.png",dpi=160)
plt.show()